# MODEL 1

# Purpose & Expected Observation

*   **Purpose**: Represents the deployable beamforming prediction pipeline using only information available before optimization decisions are made. This serves as the primary experimentally valid configuration reported in this study.
*   **Expected Observation**: Moderate ROC-AUC and F1-score due to realistic prediction difficulty under severe class imbalance. Demonstrates practical predictive capability for adaptive and real-time 6G-IoT beamforming optimization.



# Result 1. Baseline Imbalance Failure


In [ ]:
# Beamforming Optimization Classification (with One-Hot Encoding for categorical features)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Load dataset
df = pd.read_csv("6G_IoT_Beamforming_Dataset.csv")  # change path if needed


# Define feature groups (inputs) and target
network_features = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                    'Codebook Size', 'Interference Level (dB)']

environment_features = ['Obstacle Density', 'Mobility (m/s)', 'Environment_Outdoor']

device_features = ['Number of Antennas', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

target = 'Optimized'  # binary label (0 or 1)

# Specify which columns are categorical (explicit)
categorical_features = [
    "Environment_Outdoor",
    "Device Type_IoT Sensor",
    "Device Type_Smartphone"
]

# Utility function to evaluate models (with preprocessing)
def evaluate_models(X, y, feature_group_name):
    """Trains multiple classifiers on X/y and returns metrics DataFrame.
       X: pandas.DataFrame containing only the features for this group.
    """
    # Stratified split for consistent class distribution
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # Separate boolean, numeric, and categorical columns present in this feature subset
    bool_cols = [c for c in X.columns if X[c].dtype == 'bool']
    present_categorical = [c for c in categorical_features if c in X.columns and c not in bool_cols] # Exclude bool from categorical
    numeric_cols = [c for c in X.columns if c not in present_categorical and c not in bool_cols]


    # Build preprocessing pipelines
    transformers = []
    if numeric_cols:
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        transformers.append(('num', numeric_transformer, numeric_cols))

    if present_categorical:
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_transformer, present_categorical))

    if bool_cols:
        boolean_transformer = Pipeline(steps=[
             ('to_int', FunctionTransformer(lambda x: x.astype(int))), # Convert boolean to int
             ('imputer', SimpleImputer(strategy='most_frequent')) # Impute missing (if any)
        ])
        transformers.append(('bool', boolean_transformer, bool_cols))


    # ColumnTransformer (if no transformers, fallback to passthrough)
    if transformers:
        preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
        # Fit transform training and transform test
        X_train_proc = preprocessor.fit_transform(X_train)
        X_test_proc = preprocessor.transform(X_test)
    else:
        # No preprocessing needed; convert to numpy arrays
        X_train_proc = X_train.values
        X_test_proc = X_test.values


    # Define models
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
        "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "SVM (RBF)": SVC(kernel='rbf', probability=True, random_state=42),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
    }

    results = []
    for name, model in models.items():
        # Fit model
        model.fit(X_train_proc, y_train)

        # Predictions
        y_pred = model.predict(X_test_proc)

        # Probabilities for ROC-AUC (fallback to decision_function scaled if needed)
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test_proc)[:, 1]
        elif hasattr(model, "decision_function"):
            scores = model.decision_function(X_test_proc)
            y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            y_prob = y_pred  # not ideal, but keeps computation working

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        try:
            roc = roc_auc_score(y_test, y_prob)
        except Exception:
            roc = np.nan

        results.append({
            "Feature_Group": feature_group_name,
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "ROC-AUC": roc
        })

    return pd.DataFrame(results)

# Evaluate each feature group
y = df[target]

network_df = evaluate_models(df[network_features], y, "Network Parameters")
environment_df = evaluate_models(df[environment_features], y, "Environmental Factors")
device_df = evaluate_models(df[device_features], y, "Device Characteristics")

# Combine and compare
final_results = pd.concat([network_df, environment_df, device_df], ignore_index=True)

# Sort by ROC-AUC descending within each feature group
final_results = final_results.sort_values(by=["Feature_Group", "ROC-AUC"], ascending=[True, False])

# Display results
print("\n=== Beamforming Optimization Classification Results ===")
print(final_results)

# Save to CSV for analysis
final_results.to_csv("Beamforming_Classification_Results_with_OHE.csv", index=False)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(



=== Beamforming Optimization Classification Results ===
             Feature_Group                Model  Accuracy  Precision  Recall  \
18  Device Characteristics             AdaBoost  0.833333   0.000000    0.00   
14  Device Characteristics  Logistic Regression  0.833333   0.000000    0.00   
20  Device Characteristics       MLP Neural Net  0.833333   0.000000    0.00   
15  Device Characteristics        Random Forest  0.833333   0.000000    0.00   
16  Device Characteristics              XGBoost  0.833333   0.000000    0.00   
17  Device Characteristics    Gradient Boosting  0.833333   0.000000    0.00   
19  Device Characteristics            SVM (RBF)  0.833333   0.000000    0.00   
10   Environmental Factors    Gradient Boosting  0.820000   0.250000    0.04   
9    Environmental Factors              XGBoost  0.740000   0.166667    0.14   
8    Environmental Factors        Random Forest  0.743333   0.212766    0.20   
11   Environmental Factors             AdaBoost  0.833333   0.0

CHATGPT 2

# Result 2. Preprocessing Improvement

Beamforming Optimization Classification with Imbalance Handling


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from imblearn.over_sampling import SMOTE

# Load dataset
df = pd.read_csv("6G_IoT_Beamforming_Dataset.csv")  # Update path

# Feature groups
network_features = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                    'Codebook Size', 'Interference Level (dB)']

environment_features = ['Obstacle Density', 'Mobility (m/s)', 'Environment_Outdoor']

device_features = ['Number of Antennas', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

target = 'Optimized'  # Binary label

# Identify numeric columns (for scaling)
numeric_cols_master = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                       'Codebook Size', 'Interference Level (dB)',
                       'Obstacle Density', 'Mobility (m/s)',
                       'Number of Antennas']

# Explicit categorical candidates (one-hot encode these when present)
categorical_candidates = [
    'Environment_Outdoor',
    'Device Type_IoT Sensor',
    'Device Type_Smartphone'
]

# Function to evaluate models
def evaluate_models(X, y, feature_group_name):
    """
    Preprocess (impute/scale + one-hot encode), apply SMOTE, train multiple models,
    and return evaluation metrics as a DataFrame.
    """
    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # Detect numeric, categorical, and boolean columns present in this feature subset
    bool_cols = [c for c in X.columns if X[c].dtype == 'bool']
    # Ensure categorical_candidates are only treated as categorical if they are not boolean in the subset
    present_categorical = [c for c in categorical_candidates if c in X.columns and c not in bool_cols]
    # Numeric columns are those not in present_categorical or bool_cols
    present_numeric = [c for c in numeric_cols_master if c in X.columns and c not in present_categorical and c not in bool_cols]


    # Build transformers
    transformers = []
    if present_numeric:
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        transformers.append(('num', numeric_transformer, present_numeric))

    if present_categorical:
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_transformer, present_categorical))

    if bool_cols:
        # Convert boolean to float for imputer compatibility
        boolean_transformer = Pipeline(steps=[
            ('to_float', FunctionTransformer(lambda x: x.astype(float))),
            ('imputer', SimpleImputer(strategy='most_frequent')) # Impute missing (if any)
        ])
        transformers.append(('bool', boolean_transformer, bool_cols))


    # ColumnTransformer (if no transformers, pass through)
    if transformers:
        preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
        # Fit on training set and transform both train and test
        X_train_proc = preprocessor.fit_transform(X_train)
        X_test_proc = preprocessor.transform(X_test)
    else:
        # No preprocessing needed (unlikely), but convert to numpy
        X_train_proc = X_train.values
        X_test_proc = X_test.values


    # Handle class imbalance using SMOTE (on processed numeric+onehot features)
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)

    # Define models (with class_weight where appropriate)
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
        "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "SVM (RBF)": SVC(kernel='rbf', probability=True, random_state=42),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
    }

    results = []
    for name, model in models.items():
        # Fit
        model.fit(X_train_res, y_train_res)

        # Predict on the (preprocessed) test set
        y_pred = model.predict(X_test_proc)

        # Predict probabilities (or fallback to decision function)
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test_proc)[:, 1]
        elif hasattr(model, "decision_function"):
            scores = model.decision_function(X_test_proc)
            y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            # last resort: use predictions (not ideal for ROC)
            y_prob = y_pred

        # Compute metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        try:
            roc = roc_auc_score(y_test, y_prob)
        except Exception:
            roc = np.nan

        results.append({
            "Feature_Group": feature_group_name,
            "Model": name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "ROC-AUC": roc
        })

    return pd.DataFrame(results)

# Evaluate each feature group
y = df[target]

network_df = evaluate_models(df[network_features], y, "Network Parameters")
environment_df = evaluate_models(df[environment_features], y, "Environmental Factors")
device_df = evaluate_models(df[device_features], y, "Device Characteristics")


# Combine and compare
final_results = pd.concat([network_df, environment_df, device_df], ignore_index=True)

# Sort by feature group and ROC-AUC
final_results = final_results.sort_values(by=["Feature_Group", "ROC-AUC"], ascending=[True, False])

# Display results
print("\n=== Beamforming Optimization Classification Results (Balanced + Scaled + OHE) ===")
print(final_results)

# Save results
final_results.to_csv("Beamforming_Classification_Results_Balanced_OHE.csv", index=False)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(



=== Beamforming Optimization Classification Results (Balanced + Scaled + OHE) ===
             Feature_Group                Model  Accuracy  Precision  Recall  \
18  Device Characteristics             AdaBoost  0.563333   0.208633    0.58   
14  Device Characteristics  Logistic Regression  0.640000   0.221154    0.46   
19  Device Characteristics            SVM (RBF)  0.530000   0.186207    0.54   
16  Device Characteristics              XGBoost  0.453333   0.183333    0.66   
20  Device Characteristics       MLP Neural Net  0.530000   0.186207    0.54   
15  Device Characteristics        Random Forest  0.453333   0.183333    0.66   
17  Device Characteristics    Gradient Boosting  0.453333   0.183333    0.66   
13   Environmental Factors       MLP Neural Net  0.643333   0.206186    0.40   
12   Environmental Factors            SVM (RBF)  0.506667   0.173333    0.52   
9    Environmental Factors              XGBoost  0.656667   0.164557    0.26   
11   Environmental Factors           

# Result 3. Hyperparameter Optimization


Beamforming Optimization Classification with Imbalance Handling and with GridSearchCV

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Load dataset
df = pd.read_csv("6G_IoT_Beamforming_Dataset.csv")  # <- update path

# Define feature groups (inputs) and target
network_features = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                    'Codebook Size', 'Interference Level (dB)']

environment_features = ['Obstacle Density', 'Mobility (m/s)', 'Environment_Outdoor']

device_features = ['Number of Antennas', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

target_col = 'Optimized'

# Predictor and target columns (using all features for the main X, y split)
feature_cols = network_features + environment_features + device_features

X = df[feature_cols].copy()
y = df[target_col].copy()

# Global lists for numeric/categorical detection

numeric_cols_master = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                       'Codebook Size', 'Interference Level (dB)',
                       'Obstacle Density', 'Mobility (m/s)',
                       'Number of Antennas']

categorical_candidates = [
    'Environment_Outdoor',
    'Device Type_IoT Sensor',
    'Device Type_Smartphone'
]



# Small hyperparameter grids for GridSearchCV (tune as needed)
param_grids = {
    "Logistic Regression": {
        "clf__C": [0.01, 0.1, 1.0]
    },
    "Random Forest": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [None, 10]
    },
    "XGBoost": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [3, 6],
        "clf__learning_rate": [0.1, 0.01]
    },
    "Gradient Boosting": {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.1, 0.01]
    },
    "AdaBoost": {
        "clf__n_estimators": [50, 100],
        "clf__learning_rate": [1.0, 0.1]
    },
    "SVM (RBF)": {
        "clf__C": [0.1, 1.0],
        "clf__gamma": ['scale', 'auto']
    },
    "MLP Neural Net": {
        "clf__hidden_layer_sizes": [(64, 32), (128, 64)],
        "clf__alpha": [1e-4, 1e-3]
    }
}

# Function: evaluate_models with GridSearchCV (SMOTE inside pipeline)
def evaluate_models_with_grid(X, y, feature_group_name, scoring='roc_auc', n_jobs=-1):
    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )

    # Detect boolean and categorical columns present
    bool_cols = [c for c in X.columns if X[c].dtype == 'bool']
    present_categorical = [c for c in categorical_candidates if c in X.columns and c not in bool_cols]
    present_numeric = [c for c in numeric_cols_master if c in X.columns and c not in present_categorical and c not in bool_cols]

    # Build ColumnTransformer
    transformers = []
    if present_numeric:
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        transformers.append(('num', numeric_transformer, present_numeric))
    if present_categorical:
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_transformer, present_categorical))
    if bool_cols:
        boolean_transformer = Pipeline(steps=[
            ('to_float', FunctionTransformer(lambda x: x.astype(float))),
            ('imputer', SimpleImputer(strategy='most_frequent'))
        ])
        transformers.append(('bool', boolean_transformer, bool_cols))

    if transformers:
        preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
    else:
        preprocessor = None  # unlikely

    # Define base estimators
    estimators = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "SVM (RBF)": SVC(kernel='rbf', probability=True, random_state=42),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
    }

    results = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, estimator in estimators.items():
        print(f"\nRunning GridSearchCV for model: {name} on feature group: {feature_group_name}")

        # Build imbalanced pipeline: preprocessor -> SMOTE -> classifier
        steps = []
        if preprocessor is not None:
            steps.append(('preprocessor', preprocessor))
        steps.append(('smote', SMOTE(random_state=42)))
        steps.append(('clf', estimator))

        pipeline = ImbPipeline(steps=steps)

        param_grid = param_grids.get(name, {})
        # If param_grid empty, still run a quick GridSearch with default params to get CV scores
        if not param_grid:
            param_grid = {}

        gs = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring=scoring,
            cv=cv,
            n_jobs=n_jobs,
            verbose=0,
            refit=True
        )

        # Fit grid search on raw X_train (pipeline will preprocess and SMOTE internally)
        gs.fit(X_train, y_train)

        print(f" Best CV {scoring}: {gs.best_score_:.4f} | Best params: {gs.best_params_}")

        # Evaluate best estimator on held-out test set (pipeline expects raw X)
        best_pipeline = gs.best_estimator_
        y_pred = best_pipeline.predict(X_test)
        if hasattr(best_pipeline, "predict_proba"):
            y_proba = best_pipeline.predict_proba(X_test)[:, 1]
        else:
            if hasattr(best_pipeline, "decision_function"):
                scores = best_pipeline.decision_function(X_test)
                y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
            else:
                y_proba = y_pred

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        try:
            roc = roc_auc_score(y_test, y_proba)
        except Exception:
            roc = np.nan

        results.append({
            "Feature_Group": feature_group_name,
            "Model": name,
            "Best_CV_score": gs.best_score_,
            "Best_params": gs.best_params_,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "ROC-AUC": roc
        })

    return pd.DataFrame(results)


# Evaluate each group (this will run several GridSearchCVs and may take time)
y = df[target_col]

network_df = evaluate_models_with_grid(df[network_features], y, "Network Parameters", scoring='roc_auc', n_jobs=-1)
environment_df = evaluate_models_with_grid(df[environment_features], y, "Environmental Factors", scoring='roc_auc', n_jobs=-1)
device_df = evaluate_models_with_grid(df[device_features], y, "Device Characteristics", scoring='roc_auc', n_jobs=-1)


# Combine and save
final_results = pd.concat([network_df, environment_df, device_df], ignore_index=True)
final_results = final_results.sort_values(by=["Feature_Group", "ROC-AUC"], ascending=[True, False])
print("\n=== GridSearchCV results (per feature group) ===")
print(final_results)
final_results.to_csv("Beamforming_GridSearch_Results.csv", index=False)


Running GridSearchCV for model: Logistic Regression on feature group: Network Parameters
 Best CV roc_auc: 0.5305 | Best params: {'clf__C': 1.0}

Running GridSearchCV for model: Random Forest on feature group: Network Parameters
 Best CV roc_auc: 0.5437 | Best params: {'clf__max_depth': 10, 'clf__n_estimators': 200}

Running GridSearchCV for model: XGBoost on feature group: Network Parameters


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [12:12:00] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Best CV roc_auc: 0.5514 | Best params: {'clf__learning_rate': 0.01, 'clf__max_depth': 3, 'clf__n_estimators': 100}

Running GridSearchCV for model: Gradient Boosting on feature group: Network Parameters
 Best CV roc_auc: 0.5438 | Best params: {'clf__learning_rate': 0.01, 'clf__n_estimators': 200}

Running GridSearchCV for model: AdaBoost on feature group: Network Parameters
 Best CV roc_auc: 0.5262 | Best params: {'clf__learning_rate': 1.0, 'clf__n_estimators': 100}

Running GridSearchCV for model: SVM (RBF) on feature group: Network Parameters
 Best CV roc_auc: 0.5026 | Best params: {'clf__C': 0.1, 'clf__gamma': 'auto'}

Running GridSearchCV for model: MLP Neural Net on feature group: Network Parameters


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


 Best CV roc_auc: 0.4793 | Best params: {'clf__alpha': 0.001, 'clf__hidden_layer_sizes': (64, 32)}

Running GridSearchCV for model: Logistic Regression on feature group: Environmental Factors
 Best CV roc_auc: 0.4820 | Best params: {'clf__C': 0.1}

Running GridSearchCV for model: Random Forest on feature group: Environmental Factors
 Best CV roc_auc: 0.4701 | Best params: {'clf__max_depth': None, 'clf__n_estimators': 200}

Running GridSearchCV for model: XGBoost on feature group: Environmental Factors


...

 Best CV roc_auc: 0.4984 | Best params: {'clf__learning_rate': 0.01, 'clf__max_depth': 6, 'clf__n_estimators': 100}

Running GridSearchCV for model: Gradient Boosting on feature group: Environmental Factors
 Best CV roc_auc: 0.5197 | Best params: {'clf__learning_rate': 0.1, 'clf__n_estimators': 200}

Running GridSearchCV for model: AdaBoost on feature group: Environmental Factors
 Best CV roc_auc: 0.5019 | Best params: {'clf__learning_rate': 1.0, 'clf__n_estimators': 50}

Running GridSearchCV for model: SVM (RBF) on feature group: Environmental Factors
 Best CV roc_auc: 0.4818 | Best params: {'clf__C': 1.0, 'clf__gamma': 'auto'}

Running GridSearchCV for model: MLP Neural Net on feature group: Environmental Factors


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


 Best CV roc_auc: 0.4419 | Best params: {'clf__alpha': 0.001, 'clf__hidden_layer_sizes': (64, 32)}

Running GridSearchCV for model: Logistic Regression on feature group: Device Characteristics
 Best CV roc_auc: 0.5207 | Best params: {'clf__C': 1.0}

Running GridSearchCV for model: Random Forest on feature group: Device Characteristics
 Best CV roc_auc: 0.4539 | Best params: {'clf__max_depth': None, 'clf__n_estimators': 100}

Running GridSearchCV for model: XGBoost on feature group: Device Characteristics


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [12:14:42] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Best CV roc_auc: 0.4639 | Best params: {'clf__learning_rate': 0.01, 'clf__max_depth': 3, 'clf__n_estimators': 200}

Running GridSearchCV for model: Gradient Boosting on feature group: Device Characteristics
 Best CV roc_auc: 0.4581 | Best params: {'clf__learning_rate': 0.01, 'clf__n_estimators': 200}

Running GridSearchCV for model: AdaBoost on feature group: Device Characteristics
 Best CV roc_auc: 0.4999 | Best params: {'clf__learning_rate': 1.0, 'clf__n_estimators': 50}

Running GridSearchCV for model: SVM (RBF) on feature group: Device Characteristics
 Best CV roc_auc: 0.4902 | Best params: {'clf__C': 0.1, 'clf__gamma': 'auto'}

Running GridSearchCV for model: MLP Neural Net on feature group: Device Characteristics
 Best CV roc_auc: 0.4624 | Best params: {'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (64, 32)}

=== GridSearchCV results (per feature group) ===
             Feature_Group                Model  Best_CV_score  \
18  Device Characteristics             AdaBoost       

# Result 4. Threshold-aware Operational Evaluation (best)


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    xgb_available = False

# Optional sampling (SMOTE)
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    imblearn_available = True
except Exception:
    imblearn_available = False

from pandas.api.types import (
    is_integer_dtype, is_bool_dtype, is_object_dtype
)
from pandas import CategoricalDtype

RANDOM_STATE = 42

# Config
DATA_PATH = "6G_IoT_Beamforming_Dataset.csv"
TEST_SIZE = 0.2
N_SPLITS = 5  # for OOF threshold tuning

network_features = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                    'Codebook Size', 'Interference Level (dB)']

environment_features = ['Obstacle Density', 'Mobility (m/s)', 'Environment_Outdoor']

device_features = ['Number of Antennas', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

all_feature_groups = {
    "All Features": network_features + environment_features + device_features,
    "Network Parameters": network_features,
    "Environmental Factors": environment_features,
    "Device Characteristics": device_features
}

target = 'Optimized'  # binary label 0/1

# Load and basic cleaning
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

missing_cols = [c for c in all_feature_groups["All Features"] + [target] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in CSV: {missing_cols}")

# Target to binary
y_raw = df[target]
y = pd.to_numeric(y_raw, errors='coerce')
if y.isna().any():
    y_map = (
        y_raw.astype(str).str.strip().str.lower()
        .map({'1': 1, '0': 0, 'yes': 1, 'no': 0, 'true': 1, 'false': 0,
              'optimized': 1, 'not optimized': 0, 'opt': 1, 'non-opt': 0})
    )
    if y_map.isna().any():
        raise ValueError("Target column cannot be coerced to binary 0/1. Please clean the target values.")
    y = y_map
y = y.astype(int)
if set(y.unique()) - {0, 1}:
    raise ValueError(f"Target must be binary 0/1. Found: {set(y.unique())}")

pos_rate = y.mean()
print(f"Class balance: positive rate = {pos_rate:.3f} ({y.sum()} positives out of {len(y)})")

# Column type detection
def detect_column_types(X_df, explicit_binary=None, explicit_categorical=None):
    explicit_binary = set(explicit_binary or [])
    explicit_categorical = set(explicit_categorical or [])

    binary_cols, categorical_cols, numeric_cols = [], [], []

    for col in X_df.columns:
        s = X_df[col]
        if col in explicit_binary:
            binary_cols.append(col); continue
        if col in explicit_categorical:
            categorical_cols.append(col); continue

        if is_bool_dtype(s):
            binary_cols.append(col)
        elif is_object_dtype(s) or isinstance(s.dtype, CategoricalDtype):
            categorical_cols.append(col)
        else:
            if is_integer_dtype(s):
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0, 1}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
            else:
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0.0, 1.0}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
    return numeric_cols, categorical_cols, binary_cols
explicit_binary = ['Environment_Outdoor', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

# Preprocessing builders
# OneHotEncoder dense for compatibility with SMOTE
try:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('to_object', FunctionTransformer(lambda X: np.asarray(X).astype(object))),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', categorical_encoder)
])

binary_transformer = Pipeline(steps=[
    ('to_float', FunctionTransformer(lambda X: np.asarray(X).astype(float))),
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

def build_preprocessor(X_subset):
    num_cols, cat_cols, bin_cols = detect_column_types(X_subset, explicit_binary=explicit_binary)
    preproc = ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
        ('bin', binary_transformer, bin_cols)
    ], remainder='drop')
    return preproc

# Model zoo
def get_models(scale_pos_weight=1.0):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced_subsample', n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "AdaBoost": AdaBoostClassifier(n_estimators=400, learning_rate=0.05, random_state=RANDOM_STATE),
        "SVM (RBF)": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000,
                                     early_stopping=True, n_iter_no_change=10,
                                     random_state=RANDOM_STATE)
    }
    if xgb_available:
        models["XGBoost"] = XGBClassifier(
            random_state=RANDOM_STATE, n_estimators=600, learning_rate=0.05,
            max_depth=4, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.0, reg_lambda=1.0, n_jobs=-1,
            eval_metric='logloss', scale_pos_weight=scale_pos_weight
        )
    return models

# Threshold tuning via OOF predictions
def best_threshold_from_oof(pipeline, X_train, y_train, n_splits=5, metric="f1"):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros(len(y_train), dtype=float)

    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr = y_train.iloc[tr_idx]

        pipe = pipeline
        pipe.fit(X_tr, y_tr)

        if hasattr(pipe.named_steps['classifier'], "predict_proba"):
            proba = pipe.predict_proba(X_val)[:, 1]
        elif hasattr(pipe.named_steps['classifier'], "decision_function"):
            scores = pipe.decision_function(X_val)
            proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            proba = np.zeros(len(val_idx))
        oof_proba[val_idx] = proba

    y_true = y_train.values
    # Evaluate thresholds
    thresholds = np.unique(np.round(oof_proba, 4))
    if len(thresholds) < 50:
        thresholds = np.linspace(0.05, 0.95, 91)

    best_thr, best_val = 0.5, -1
    for thr in thresholds:
        y_hat = (oof_proba >= thr).astype(int)
        if metric == "f1":
            val = f1_score(y_true, y_hat, zero_division=0)
        elif metric == "recall":
            val = recall_score(y_true, y_hat, zero_division=0)
        else:
            val = f1_score(y_true, y_hat, zero_division=0)
        if val > best_val:
            best_val, best_thr = val, thr
    return float(best_thr), float(best_val)

# Evaluate function per group
def evaluate_group(df, features, y, group_name):
    X = df[features].copy()

    preproc = build_preprocessor(X)

    pos_rate = y.mean()
    neg_rate = 1 - pos_rate
    scale_pos_weight = (neg_rate / pos_rate) if 0 < pos_rate < 1 else 1.0

    models = get_models(scale_pos_weight=scale_pos_weight)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    use_smote = imblearn_available and pos_rate < 0.35  # enable if quite imbalanced
    results = []

    for name, clf in models.items():
        if use_smote:
            pipe_base = ImbPipeline(steps=[('preprocessor', preproc), ('smote', SMOTE(random_state=RANDOM_STATE)), ('classifier', clf)])
        else:
            pipe_base = Pipeline(steps=[('preprocessor', preproc), ('classifier', clf)])

        # Find threshold using OOF on training set (no test leakage)
        thr, oof_score = best_threshold_from_oof(pipe_base, X_train, y_train, n_splits=N_SPLITS, metric="f1")

        # Fit on full training and evaluate
        pipe_base.fit(X_train, y_train)

        # Probas for AUC metrics
        if hasattr(pipe_base.named_steps['classifier'], "predict_proba"):
            y_proba_test = pipe_base.predict_proba(X_test)[:, 1]
        elif hasattr(pipe_base.named_steps['classifier'], "decision_function"):
            scores = pipe_base.decision_function(X_test)
            y_proba_test = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            y_proba_test = np.zeros_like(y_test, dtype=float)


        # Default 0.5
        y_pred_05 = (y_proba_test >= 0.5).astype(int)
        # Tuned threshold
        y_pred_thr = (y_proba_test >= thr).astype(int)


        # Metrics
        def safe_metrics(y_true, y_pred, y_proba):
            acc = accuracy_score(y_true, y_pred)
            prec = precision_score(y_true, y_pred, zero_division=0)
            rec = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            roc = roc_auc_score(y_true, y_proba) if y_proba is not None and len(np.unique(y_true)) == 2 else np.nan
            pr  = average_precision_score(y_true, y_proba) if y_proba is not None and len(np.unique(y_true)) == 2 else np.nan
            return acc, prec, rec, f1, roc, pr

        acc05, prec05, rec05, f105, roc, prauc = safe_metrics(y_test, y_pred_05, y_proba_test)
        acct,   prect,   rect,   f1t,  _,   _  = safe_metrics(y_test, y_pred_thr, y_proba_test)


        results.append({
            "Feature_Group": group_name,
            "Model": name,
            "Use_SMOTE": use_smote,
            "Threshold_OOF_F1": round(thr, 3),
            "OOF_F1_at_Thr": round(oof_score, 3),
            "Accuracy@0.5": acc05,
            "Precision@0.5": prec05,
            "Recall@0.5": rec05,
            "F1@0.5": f105,
            "Accuracy@Tuned": acct,
            "Precision@Tuned": prect,
            "Recall@Tuned": rect,
            "F1@Tuned": f1t,
            "ROC-AUC": roc,
            "PR-AUC": prauc
        })


    return pd.DataFrame(results)

# Run evaluations
all_results = []
for group_name, features in all_feature_groups.items():
    res = evaluate_group(df, features, y, group_name)
    all_results.append(res)


final_results = pd.concat(all_results, ignore_index=True)

# Sort for readability: by Feature_Group then F1@Tuned desc
final_results = final_results.sort_values(by=["Feature_Group", "F1@Tuned"], ascending=[True, False])

print("\n=== Beamforming Optimization Classification Results (with preprocessing, imbalance handling, and threshold tuning) ===")
print(final_results)

final_results.to_csv("Beamforming_Classification_Results_Improved.csv", index=False)

Class balance: positive rate = 0.168 (168 positives out of 1000)

=== Beamforming Optimization Classification Results (with preprocessing, imbalance handling, and threshold tuning) ===
             Feature_Group                Model  Use_SMOTE  Threshold_OOF_F1  \
5             All Features       MLP Neural Net       True             0.002   
2             All Features    Gradient Boosting       True             0.044   
6             All Features              XGBoost       True             0.002   
3             All Features             AdaBoost       True             0.282   
4             All Features            SVM (RBF)       True             0.005   
0             All Features  Logistic Regression       True             0.385   
1             All Features        Random Forest       True             0.128   
21  Device Characteristics  Logistic Regression       True             0.388   
22  Device Characteristics        Random Forest       True             0.221   
23  Device Char

# Result 5. Over-calibration (Cautionary Case)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, FunctionTransformer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import (
accuracy_score, precision_score, recall_score, f1_score,
roc_auc_score, average_precision_score
)
from sklearn.calibration import CalibratedClassifierCV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    xgb_available = False

# Optional sampling (SMOTE and variants)
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imbleberan.over_sampling import SMOTE
    from imblearn.combine import SMOTETomek
    imblearn_available = True
except Exception:
    imblearn_available = False

from pandas.api.types import is_integer_dtype, is_bool_dtype, is_object_dtype
from pandas import CategoricalDtype

RANDOM_STATE = 42

# Config
DATA_PATH = "6G_IoT_Beamforming_Dataset.csv"
TEST_SIZE = 0.2
N_SPLITS = 5
THRESH_OBJECTIVE = "f1"  # or "f2" for recall emphasis, or "recall"

USE_INTERACTIONS = True            # Add 2-way numeric interactions (interaction_only)
USE_CALIBRATION = True             # Calibrate probabilities for tree ensembles
USE_SMOTE_FOR_LINEAR = True        # Apply SMOTE to LR/SVM/MLP
USE_SMOTE_FOR_TREES = False        # Avoid SMOTE for tree ensembles by default
SMOTE_STRATEGY = 0.5               # Oversample minority to 50% of majority (milder than 1.0)
USE_SMOTETOMEK = False             # Alternative to SMOTE to reduce noisy border samples

network_features = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
'Codebook Size', 'Interference Level (dB)']

environment_features = ['Obstacle Density', 'Mobility (m/s)', 'Environment_Outdoor']

device_features = ['Number of Antennas', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

all_feature_groups = {
"All Features": network_features + environment_features + device_features,
"Network Parameters": network_features,
"Environmental Factors": environment_features,
"Device Characteristics": device_features
}

target = 'Optimized'

# Load and clean
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

missing_cols = [c for c in all_feature_groups["All Features"] + [target] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Target to binary
y_raw = df[target]
y = pd.to_numeric(y_raw, errors='coerce')
if y.isna().any():
    y_map = (
        y_raw.astype(str).str.strip().str.lower()
        .map({'1': 1, '0': 0, 'yes': 1, 'no': 0, 'true': 1, 'false': 0,
              'optimized': 1, 'not optimized': 0, 'opt': 1, 'non-opt': 0})
    )
    if y_map.isna().any():
        raise ValueError("Target cannot be coerced to 0/1. Please clean labels.")
    y = y_map
y = y.astype(int)
if set(y.unique()) - {0, 1}:
    raise ValueError(f"Target must be binary 0/1. Found: {set(y.unique())}")

pos_rate = y.mean()
print(f"Class balance: positive rate = {pos_rate:.3f} ({y.sum()} positives out of {len(y)})")


# Column type detection
def detect_column_types(X_df, explicit_binary=None, explicit_categorical=None):
    explicit_binary = set(explicit_binary or [])
    explicit_categorical = set(explicit_categorical or [])
    binary_cols, categorical_cols, numeric_cols = [], [], []
    for col in X_df.columns:
        s = X_df[col]
        if col in explicit_binary:
            binary_cols.append(col); continue
        if col in explicit_categorical:
            categorical_cols.append(col); continue
        if is_bool_dtype(s):
            binary_cols.append(col)
        elif is_object_dtype(s) or isinstance(s.dtype, CategoricalDtype):
            categorical_cols.append(col)
        else:
            if is_integer_dtype(s):
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0, 1}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
            else:
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0.0, 1.0}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
    return numeric_cols, categorical_cols, binary_cols

explicit_binary = ['Environment_Outdoor', 'Device Type_IoT Sensor', 'Device Type_Smartphone']


# Preprocessing builders
try:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

numeric_base = [('imputer', SimpleImputer(strategy='median'))]
if USE_INTERACTIONS:
    numeric_base.append(('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)))
numeric_base.append(('scaler', RobustScaler()))
numeric_transformer = Pipeline(steps=numeric_base)

categorical_transformer = Pipeline(steps=[
('to_object', FunctionTransformer(lambda X: np.asarray(X).astype(object))),
('imputer', SimpleImputer(strategy='most_frequent')),
('onehot', categorical_encoder)
])

binary_transformer = Pipeline(steps=[
('to_float', FunctionTransformer(lambda X: np.asarray(X).astype(float))),
('imputer', SimpleImputer(strategy='most_frequent'))
])

def build_preprocessor(X_subset):
    num_cols, cat_cols, bin_cols = detect_column_types(X_subset, explicit_binary=explicit_binary)
    return ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
    ('bin', binary_transformer, bin_cols)
    ], remainder='drop')


# Models and wrappers
def get_models(scale_pos_weight=1.0):
    models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    "SVM (RBF)": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000,
    early_stopping=True, n_iter_no_change=10,
    random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
    n_estimators=600, max_depth=None, min_samples_leaf=2,
    random_state=RANDOM_STATE, class_weight='balanced_subsample', n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(n_estimators=600, learning_rate=0.05, random_state=RANDOM_STATE),
    }
    if xgb_available:
        models["XGBoost"] = XGBClassifier(
        random_state=RANDOM_STATE, n_estimators=800, learning_rate=0.05,
        max_depth=4, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.0, reg_lambda=1.0, n_jobs=-1,
        eval_metric='logloss', scale_pos_weight=scale_pos_weight
        )
    return models

def maybe_calibrated(estimator, name):
    if USE_CALIBRATION and name in {"Random Forest", "Gradient Boosting", "AdaBoost", "XGBoost"}:
    # Isotonic needs more data; sigmoid is safer. Use sigmoid here.
        return CalibratedClassifierCV(estimator, method='sigmoid', cv=3)
    return estimator

def build_pipeline_for_model(name, clf, X_subset, pos_rate):
    preproc = build_preprocessor(X_subset)
    # Decide on sampling for this model
    use_sampler = False
    if imblearn_available:
        if name in {"Logistic Regression", "SVM (RBF)", "MLP Neural Net"} and USE_SMOTE_FOR_LINEAR:
            use_sampler = True
        if name in {"Random Forest", "Gradient Boosting", "AdaBoost", "XGBoost"} and USE_SMOTE_FOR_TREES:
            use_sampler = True

    estimator = maybe_calibrated(clf, name)

    if use_sampler:
        sampler = SMOTETomek(random_state=RANDOM_STATE, sampling_strategy=SMOTE_STRATEGY) if USE_SMOTETOMEK else \
                  SMOTE(random_state=RANDOM_STATE, sampling_strategy=SMOTE_STRATEGY)
        pipe = ImbPipeline(steps=[('preprocessor', preproc), ('sampler', sampler), ('classifier', estimator)])
    else:
        pipe = Pipeline(steps=[('preprocessor', preproc), ('classifier', estimator)])
    return pipe


# Threshold tuning (supports F1 or F-beta)
def fbeta_from_preds(y_true, y_pred, beta=1.0):
    if beta == 1.0:
        return f1_score(y_true, y_pred, zero_division=0)
    # Manual F-beta
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    if prec == 0 and rec == 0:
        return 0.0
    b2 = beta * beta
    return (1 + b2) * (prec * rec) / (b2 * prec + rec + 1e-12)

def best_threshold_from_oof(pipeline, X_train, y_train, n_splits=5, objective="f1"):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros(len(y_train), dtype=float)
    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr = y_train.iloc[tr_idx]
        pipe = pipeline
        pipe.fit(X_tr, y_tr)
        if hasattr(pipe.named_steps['classifier'], "predict_proba"):
            proba = pipe.predict_proba(X_val)[:, 1]
        elif hasattr(pipe.named_steps['classifier'], "decision_function"):
            scores = pipe.decision_function(X_val)
            proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            proba = np.zeros(len(val_idx))
        oof_proba[val_idx] = proba

    y_true = y_train.values
    thresholds = np.linspace(0.01, 0.99, 99)
    best_thr, best_val = 0.5, -1.0
    beta = 2.0 if objective == "f2" else 1.0
    for thr in thresholds:
        y_hat = (oof_proba >= thr).astype(int)
        score = recall_score(y_true, y_hat, zero_division=0) if objective == "recall" else fbeta_from_preds(y_true, y_hat, beta)
        if score > best_val:
            best_val, best_thr = score, thr
    return float(best_thr), float(best_val)

# Evaluation per group
def evaluate_group(df, features, y, group_name):
    X = df[features].copy()
    pos_rate = y.mean()
    neg_rate = 1 - pos_rate
    spw = (neg_rate / pos_rate) if 0 < pos_rate < 1 else 1.0

    models = get_models(scale_pos_weight=spw)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    results = []

    for name, clf in models.items():
        pipe = build_pipeline_for_model(name, clf, X, pos_rate)

        thr, oof_score = best_threshold_from_oof(pipe, X_train, y_train, n_splits=N_SPLITS, objective=THRESH_OBJECTIVE)

        pipe.fit(X_train, y_train)

        if hasattr(pipe.named_steps['classifier'], "predict_proba"):
            y_proba_test = pipe.predict_proba(X_test)[:, 1]
        elif hasattr(pipe.named_steps['classifier'], "decision_function"):
            scores = pipe.decision_function(X_test)
            y_proba_test = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            y_proba_test = np.zeros_like(y_test, dtype=float)

        y_pred_05 = (y_proba_test >= 0.5).astype(int)
        y_pred_thr = (y_proba_test >= thr).astype(int)

        def metrics(y_true, y_pred, y_proba):
            acc = accuracy_score(y_true, y_pred)
            prec = precision_score(y_true, y_pred, zero_division=0)
            rec = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            roc = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) == 2 else np.nan
            pr  = average_precision_score(y_true, y_proba) if len(np.unique(y_true)) == 2 else np.nan
            return acc, prec, rec, f1, roc, pr

        acc05, prec05, rec05, f105, roc, prauc = metrics(y_test, y_pred_05, y_proba_test)
        acct, prect, rect, f1t, _, _ = metrics(y_test, y_pred_thr, y_proba_test)

        # Whether this model used a sampler
        use_smote_flag = isinstance(pipe, ImbPipeline) if imblearn_available else False

        results.append({
            "Feature_Group": group_name,
            "Model": name,
            "Use_SMOTE": use_smote_flag,
            "Threshold_OOF_F1": round(thr, 3),
            "OOF_F1_at_Thr": round(oof_score, 3),
            "Accuracy@0.5": acc05,
            "Precision@0.5": prec05,
            "Recall@0.5": rec05,
            "F1@0.5": f105,
            "Accuracy@Tuned": acct,
            "Precision@Tuned": prect,
            "Recall@Tuned": rect,
            "F1@Tuned": f1t,
            "ROC-AUC": roc,
            "PR-AUC": prauc
        })

    return pd.DataFrame(results)

# Run
all_results = []
for group_name, features in all_feature_groups.items():
    res = evaluate_group(df, features, y, group_name)
    all_results.append(res)

final_results = pd.concat(all_results, ignore_index=True)
final_results = final_results.sort_values(by=["Feature_Group", "F1@Tuned"], ascending=[True, False])

print("\n=== Beamforming Optimization Classification Results (improved preprocessing, calibrated, milder sampling, tuned threshold) ===")
print(final_results)
final_results.to_csv("Beamforming_Classification_Results_Improved_v2.csv", index=False)

Class balance: positive rate = 0.168 (168 positives out of 1000)

=== Beamforming Optimization Classification Results (improved preprocessing, calibrated, milder sampling, tuned threshold) ===
             Feature_Group                Model  Use_SMOTE  Threshold_OOF_F1  \
3             All Features        Random Forest      False              0.14   
0             All Features  Logistic Regression      False              0.15   
1             All Features            SVM (RBF)      False              0.12   
4             All Features    Gradient Boosting      False              0.12   
5             All Features             AdaBoost      False              0.14   
6             All Features              XGBoost      False              0.11   
2             All Features       MLP Neural Net      False              0.14   
21  Device Characteristics  Logistic Regression      False              0.01   
22  Device Characteristics            SVM (RBF)      False              0.01   
23  Dev

# Result 4 (Best Option) with More Models (Values was used in the paper)



In [7]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    xgb_available = False
try:
    from lightgbm import LGBMClassifier
    lgbm_available = True
except Exception:
    lgbm_available = False

# Optional sampling (SMOTE)
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    imblearn_available = True
except Exception:
    imblearn_available = False

from pandas.api.types import (
    is_integer_dtype, is_bool_dtype, is_object_dtype
)
from pandas import CategoricalDtype

RANDOM_STATE = 42

# Config
DATA_PATH = "6G_IoT_Beamforming_Dataset.csv"
TEST_SIZE = 0.2
N_SPLITS = 5  # for OOF threshold tuning

network_features = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                    'Codebook Size', 'Interference Level (dB)']

environment_features = ['Obstacle Density', 'Mobility (m/s)', 'Environment_Outdoor']

device_features = ['Number of Antennas', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

vision_features = ['SIFT Keypoints']

all_feature_groups = {
    "All Features": network_features + environment_features + device_features + vision_features,
    "Network Parameters": network_features,
    "Environmental Factors": environment_features,
    "Device Characteristics": device_features,
    "Vision Features": vision_features
}

target = 'Optimized'  # binary label 0/1

# Load and basic cleaning
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

missing_cols = [c for c in all_feature_groups["All Features"] + [target] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in CSV: {missing_cols}")

# Target to binary
y_raw = df[target]
y = pd.to_numeric(y_raw, errors='coerce')
if y.isna().any():
    y_map = (
        y_raw.astype(str).str.strip().str.lower()
        .map({'1': 1, '0': 0, 'yes': 1, 'no': 0, 'true': 1, 'false': 0,
              'optimized': 1, 'not optimized': 0, 'opt': 1, 'non-opt': 0})
    )
    if y_map.isna().any():
        raise ValueError("Target column cannot be coerced to binary 0/1. Please clean the target values.")
    y = y_map
y = y.astype(int)
if set(y.unique()) - {0, 1}:
    raise ValueError(f"Target must be binary 0/1. Found: {set(y.unique())}")

pos_rate = y.mean()
print(f"Class balance: positive rate = {pos_rate:.3f} ({y.sum()} positives out of {len(y)})")

# Column type detection
def detect_column_types(X_df, explicit_binary=None, explicit_categorical=None):
    explicit_binary = set(explicit_binary or [])
    explicit_categorical = set(explicit_categorical or [])

    binary_cols, categorical_cols, numeric_cols = [], [], []

    for col in X_df.columns:
        s = X_df[col]
        if col in explicit_binary:
            binary_cols.append(col); continue
        if col in explicit_categorical:
            categorical_cols.append(col); continue

        if is_bool_dtype(s):
            binary_cols.append(col)
        elif is_object_dtype(s) or isinstance(s.dtype, CategoricalDtype):
            categorical_cols.append(col)
        else:
            if is_integer_dtype(s):
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0, 1}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
            else:
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0.0, 1.0}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
    return numeric_cols, categorical_cols, binary_cols
explicit_binary = ['Environment_Outdoor', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

# Preprocessing builders
# OneHotEncoder dense for compatibility with SMOTE
try:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('to_object', FunctionTransformer(lambda X: np.asarray(X).astype(object))),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', categorical_encoder)
])

binary_transformer = Pipeline(steps=[
    ('to_float', FunctionTransformer(lambda X: np.asarray(X).astype(float))),
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

def build_preprocessor(X_subset):
    num_cols, cat_cols, bin_cols = detect_column_types(X_subset, explicit_binary=explicit_binary)
    preproc = ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
        ('bin', binary_transformer, bin_cols)
    ], remainder='drop')
    return preproc

# Model zoo
def get_models(scale_pos_weight=1.0):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(
            n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced_subsample', n_jobs=-1
        ),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "AdaBoost": AdaBoostClassifier(n_estimators=400, learning_rate=0.05, random_state=RANDOM_STATE),
        "SVM (RBF)": SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000,
                                     early_stopping=True, n_iter_no_change=10,
                                     random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        "KNeighbors": KNeighborsClassifier(n_neighbors=5),
        "Extra Trees": ExtraTreesClassifier(n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced_subsample', n_jobs=-1),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE)
    }
    if xgb_available:
        models["XGBoost"] = XGBClassifier(
            random_state=RANDOM_STATE, n_estimators=600, learning_rate=0.05,
            max_depth=4, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.0, reg_lambda=1.0, n_jobs=-1,
            eval_metric='logloss', scale_pos_weight=scale_pos_weight
        )
    if lgbm_available:
        models["LightGBM"] = LGBMClassifier(
            random_state=RANDOM_STATE, n_estimators=600, learning_rate=0.05,
            num_leaves=31, max_depth=-1, colsample_bytree=0.8, subsample=0.8,
            reg_alpha=0.0, reg_lambda=1.0, n_jobs=-1,
            scale_pos_weight=scale_pos_weight
        )
    return models

# Threshold tuning via OOF predictions
def best_threshold_from_oof(pipeline, X_train, y_train, n_splits=5, metric="f1"):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros(len(y_train), dtype=float)

    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr = y_train.iloc[tr_idx]

        pipe = pipeline
        pipe.fit(X_tr, y_tr)

        if hasattr(pipe.named_steps['classifier'], "predict_proba"):
            proba = pipe.predict_proba(X_val)[:, 1]
        elif hasattr(pipe.named_steps['classifier'], "decision_function"):
            scores = pipe.decision_function(X_val)
            proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            proba = np.zeros(len(val_idx))
        oof_proba[val_idx] = proba

    y_true = y_train.values
    # Evaluate thresholds
    thresholds = np.unique(np.round(oof_proba, 4))
    if len(thresholds) < 50:
        thresholds = np.linspace(0.05, 0.95, 91)

    best_thr, best_val = 0.5, -1
    for thr in thresholds:
        y_hat = (oof_proba >= thr).astype(int)
        if metric == "f1":
            val = f1_score(y_true, y_hat, zero_division=0)
        elif metric == "recall":
            val = recall_score(y_true, y_hat, zero_division=0)
        else:
            val = f1_score(y_true, y_hat, zero_division=0)
        if val > best_val:
            best_val, best_thr = val, thr
    return float(best_thr), float(best_val)

# Evaluate function per group
def evaluate_group(df, features, y, group_name):
    X = df[features].copy()

    preproc = build_preprocessor(X)

    pos_rate = y.mean()
    neg_rate = 1 - pos_rate
    scale_pos_weight = (neg_rate / pos_rate) if 0 < pos_rate < 1 else 1.0

    models = get_models(scale_pos_weight=scale_pos_weight)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    use_smote = imblearn_available and pos_rate < 0.35  # enable if quite imbalanced
    results = []

    for name, clf in models.items():
        if use_smote:
            pipe_base = ImbPipeline(steps=[('preprocessor', preproc), ('smote', SMOTE(random_state=RANDOM_STATE)), ('classifier', clf)])
        else:
            pipe_base = Pipeline(steps=[('preprocessor', preproc), ('classifier', clf)])

        # Find threshold using OOF on training set (no test leakage)
        thr, oof_score = best_threshold_from_oof(pipe_base, X_train, y_train, n_splits=N_SPLITS, metric="f1")

        # Fit on full training and evaluate
        pipe_base.fit(X_train, y_train)

        # Probas for AUC metrics
        if hasattr(pipe_base.named_steps['classifier'], "predict_proba"):
            y_proba_test = pipe_base.predict_proba(X_test)[:, 1]
        elif hasattr(pipe.named_steps['classifier'], "decision_function"):
            scores = pipe.decision_function(X_test)
            y_proba_test = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            y_proba_test = np.zeros_like(y_test, dtype=float)


        # Default 0.5
        y_pred_05 = (y_proba_test >= 0.5).astype(int)
        # Tuned threshold
        y_pred_thr = (y_proba_test >= thr).astype(int)


        # Metrics
        def safe_metrics(y_true, y_pred, y_proba):
            acc = accuracy_score(y_true, y_pred)
            prec = precision_score(y_true, y_pred, zero_division=0)
            rec = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            roc = roc_auc_score(y_true, y_proba) if y_proba is not None and len(np.unique(y_true)) == 2 else np.nan
            pr  = average_precision_score(y_true, y_proba) if y_proba is not None and len(np.unique(y_true)) == 2 else np.nan
            return acc, prec, rec, f1, roc, pr

        acc05, prec05, rec05, f105, roc, prauc = safe_metrics(y_test, y_pred_05, y_proba_test)
        acct,   prect,   rect,   f1t,  _,   _  = safe_metrics(y_test, y_pred_thr, y_proba_test)


        results.append({
            "Feature_Group": group_name,
            "Model": name,
            "Use_SMOTE": use_smote,
            "Threshold_OOF_F1": round(thr, 3),
            "OOF_F1_at_Thr": round(oof_score, 3),
            "Accuracy@0.5": acc05,
            "Precision@0.5": prec05,
            "Recall@0.5": rec05,
            "F1@0.5": f105,
            "Accuracy@Tuned": acct,
            "Precision@Tuned": prect,
            "Recall@Tuned": rect,
            "F1@Tuned": f1t,
            "ROC-AUC": roc,
            "PR-AUC": prauc
        })


    return pd.DataFrame(results)

# Run evaluations
all_results = []
for group_name, features in all_feature_groups.items():
    res = evaluate_group(df, features, y, group_name)
    all_results.append(res)


final_results = pd.concat(all_results, ignore_index=True)

# Sort for readability: by Feature_Group then F1@Tuned desc
final_results = final_results.sort_values(by=["Feature_Group", "F1@Tuned"], ascending=[True, False])

print("\n=== Beamforming Optimization Classification Results (with preprocessing, imbalance handling, and threshold tuning) ===")
print(final_results)

final_results.to_csv("Beamforming_Classification_Results_Improved.csv", index=False)


Class balance: positive rate = 0.168 (168 positives out of 1000)
[LightGBM] [Info] Number of positive: 532, number of negative: 532
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012289 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1996
[LightGBM] [Info] Number of data points in the train set: 1064, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 533, number of negative: 533
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000196 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1989
[LightGBM] [Info] Number of data points in the train set: 1066, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 533, number of negative: 533
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000040 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 252
[LightGBM] [Info] Number of data points in the train set: 1066, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 666, number of negative: 666
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014756 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 252
[LightGBM] [Info] Number of data points in the train set: 1332, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

=== Beamforming Optimization Classification Results (with preprocessing, imbalance handling, and threshold tuning) ===
             Feature_Group                 Model  Use_SMOTE  Threshold_OOF_F1  \
5             All Features        MLP Neural Net       True        

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# BASELINE Pipeline with SMOTE & GridSearchCV

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Load dataset
df = pd.read_csv("6G_IoT_Beamforming_Dataset.csv")  # <- update path

# Define feature groups (inputs) and target

network_features = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                    'Codebook Size', 'Interference Level (dB)']

environment_features = ['Obstacle Density', 'Mobility (m/s)', 'Environment_Outdoor']

device_features = ['Number of Antennas', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

vision_features = ['SIFT Keypoints']

all_feature_groups = {
    "All Features": network_features + environment_features + device_features + vision_features,
    "Network Parameters": network_features,
    "Environmental Factors": environment_features,
    "Device Characteristics": device_features,
    "Vision Features": vision_features
}

target_col = 'Optimized'

# Predictor and target columns (using all features for the main X, y split)
feature_cols = network_features + environment_features + device_features + vision_features

X = df[feature_cols].copy()
y = df[target_col].copy()

# Global lists for numeric/categorical detection

numeric_cols_master = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                       'Codebook Size', 'Interference Level (dB)',
                       'Obstacle Density', 'Mobility (m/s)',
                       'Number of Antennas']

categorical_candidates = [
    'Environment_Outdoor',
    'Device Type_IoT Sensor',
    'Device Type_Smartphone'
]


# Small hyperparameter grids for GridSearchCV (tune as needed)
param_grids = {
    "Logistic Regression": {
        "clf__C": [0.01, 0.1, 1.0]
    },
    "Random Forest": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [None, 10]
    },
    "XGBoost": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [3, 6],
        "clf__learning_rate": [0.1, 0.01]
    },
    "Gradient Boosting": {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.1, 0.01]
    },
    "AdaBoost": {
        "clf__n_estimators": [50, 100],
        "clf__learning_rate": [1.0, 0.1]
    },
    "SVM (RBF)": {
        "clf__C": [0.1, 1.0],
        "clf__gamma": ['scale', 'auto']
    },
    "MLP Neural Net": {
        "clf__hidden_layer_sizes": [(64, 32), (128, 64)],
        "clf__alpha": [1e-4, 1e-3]
    }
}

# Function: evaluate_models with GridSearchCV (SMOTE inside pipeline)
def evaluate_models_with_grid(X, y, feature_group_name, scoring='roc_auc', n_jobs=-1):
    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )

    # Detect boolean and categorical columns present
    bool_cols = [c for c in X.columns if X[c].dtype == 'bool']
    present_categorical = [c for c in categorical_candidates if c in X.columns and c not in bool_cols]
    present_numeric = [c for c in numeric_cols_master if c in X.columns and c not in present_categorical and c not in bool_cols]

    # Build ColumnTransformer
    transformers = []
    if present_numeric:
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        transformers.append(('num', numeric_transformer, present_numeric))
    if present_categorical:
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_transformer, present_categorical))
    if bool_cols:
        boolean_transformer = Pipeline(steps=[
            ('to_float', FunctionTransformer(lambda x: x.astype(float))),
            ('imputer', SimpleImputer(strategy='most_frequent'))
        ])
        transformers.append(('bool', boolean_transformer, bool_cols))

    if transformers:
        preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
    else:
        preprocessor = None  # unlikely

    # Define base estimators
    estimators = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "SVM (RBF)": SVC(kernel='rbf', probability=True, random_state=42),
        "MLP Neural Net": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
    }

    results = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, estimator in estimators.items():
        print(f"\nRunning GridSearchCV for model: {name} on feature group: {feature_group_name}")

        # Build imbalanced pipeline: preprocessor -> SMOTE -> classifier
        steps = []
        if preprocessor is not None:
            steps.append(('preprocessor', preprocessor))
        steps.append(('smote', SMOTE(random_state=42)))
        steps.append(('clf', estimator))

        pipeline = ImbPipeline(steps=steps)

        param_grid = param_grids.get(name, {})
        # If param_grid empty, still run a quick GridSearch with default params to get CV scores
        if not param_grid:
            param_grid = {}

        gs = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring=scoring,
            cv=cv,
            n_jobs=n_jobs,
            verbose=0,
            refit=True
        )

        # Fit grid search on raw X_train (pipeline will preprocess and SMOTE internally)
        gs.fit(X_train, y_train)

        print(f" Best CV {scoring}: {gs.best_score_:.4f} | Best params: {gs.best_params_}")

        # Evaluate best estimator on held-out test set (pipeline expects raw X)
        best_pipeline = gs.best_estimator_
        y_pred = best_pipeline.predict(X_test)
        if hasattr(best_pipeline, "predict_proba"):
            y_proba = best_pipeline.predict_proba(X_test)[:, 1]
        else:
            if hasattr(best_pipeline, "decision_function"):
                scores = best_pipeline.decision_function(X_test)
                y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
            else:
                y_proba = y_pred

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        try:
            roc = roc_auc_score(y_test, y_proba)
        except Exception:
            roc = np.nan

        results.append({
            "Feature_Group": feature_group_name,
            "Model": name,
            "Best_CV_score": gs.best_score_,
            "Best_params": gs.best_params_,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "ROC-AUC": roc
        })

    return pd.DataFrame(results)


# Evaluate each group (this will run several GridSearchCVs and may take time)
y = df[target_col]

network_df = evaluate_models_with_grid(df[network_features], y, "Network Parameters", scoring='roc_auc', n_jobs=-1)
environment_df = evaluate_models_with_grid(df[environment_features], y, "Environmental Factors", scoring='roc_auc', n_jobs=-1)
device_df = evaluate_models_with_grid(df[device_features], y, "Device Characteristics", scoring='roc_auc', n_jobs=-1)
vision_df = evaluate_models_with_grid(df[vision_features], y, "vision Characteristics", scoring='roc_auc', n_jobs=-1)

# Combine and save
final_results = pd.concat([network_df, environment_df, device_df, vision_df], ignore_index=True)
final_results = final_results.sort_values(by=["Feature_Group", "ROC-AUC"], ascending=[True, False])
print("\n=== GridSearchCV results (per feature group) ===")
print(final_results)
final_results.to_csv("Beamforming_GridSearch_Results.csv", index=False)


Running GridSearchCV for model: Logistic Regression on feature group: Network Parameters
 Best CV roc_auc: 0.5305 | Best params: {'clf__C': 1.0}

Running GridSearchCV for model: Random Forest on feature group: Network Parameters
 Best CV roc_auc: 0.5437 | Best params: {'clf__max_depth': 10, 'clf__n_estimators': 200}

Running GridSearchCV for model: XGBoost on feature group: Network Parameters


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:34:15] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Best CV roc_auc: 0.5514 | Best params: {'clf__learning_rate': 0.01, 'clf__max_depth': 3, 'clf__n_estimators': 100}

Running GridSearchCV for model: Gradient Boosting on feature group: Network Parameters
 Best CV roc_auc: 0.5438 | Best params: {'clf__learning_rate': 0.01, 'clf__n_estimators': 200}

Running GridSearchCV for model: AdaBoost on feature group: Network Parameters
 Best CV roc_auc: 0.5262 | Best params: {'clf__learning_rate': 1.0, 'clf__n_estimators': 100}

Running GridSearchCV for model: SVM (RBF) on feature group: Network Parameters
 Best CV roc_auc: 0.5026 | Best params: {'clf__C': 0.1, 'clf__gamma': 'auto'}

Running GridSearchCV for model: MLP Neural Net on feature group: Network Parameters


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


 Best CV roc_auc: 0.4793 | Best params: {'clf__alpha': 0.001, 'clf__hidden_layer_sizes': (64, 32)}

Running GridSearchCV for model: Logistic Regression on feature group: Environmental Factors
 Best CV roc_auc: 0.4820 | Best params: {'clf__C': 0.1}

Running GridSearchCV for model: Random Forest on feature group: Environmental Factors
 Best CV roc_auc: 0.4701 | Best params: {'clf__max_depth': None, 'clf__n_estimators': 200}

Running GridSearchCV for model: XGBoost on feature group: Environmental Factors


...

 Best CV roc_auc: 0.4419 | Best params: {'clf__alpha': 0.001, 'clf__hidden_layer_sizes': (64, 32)}

Running GridSearchCV for model: Logistic Regression on feature group: Device Characteristics
 Best CV roc_auc: 0.5207 | Best params: {'clf__C': 1.0}

Running GridSearchCV for model: Random Forest on feature group: Device Characteristics
 Best CV roc_auc: 0.4539 | Best params: {'clf__max_depth': None, 'clf__n_estimators': 100}

Running GridSearchCV for model: XGBoost on feature group: Device Characteristics


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:37:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Best CV roc_auc: 0.4639 | Best params: {'clf__learning_rate': 0.01, 'clf__max_depth': 3, 'clf__n_estimators': 200}

Running GridSearchCV for model: Gradient Boosting on feature group: Device Characteristics
 Best CV roc_auc: 0.4581 | Best params: {'clf__learning_rate': 0.01, 'clf__n_estimators': 200}

Running GridSearchCV for model: AdaBoost on feature group: Device Characteristics
 Best CV roc_auc: 0.4999 | Best params: {'clf__learning_rate': 1.0, 'clf__n_estimators': 50}

Running GridSearchCV for model: SVM (RBF) on feature group: Device Characteristics
 Best CV roc_auc: 0.4902 | Best params: {'clf__C': 0.1, 'clf__gamma': 'auto'}

Running GridSearchCV for model: MLP Neural Net on feature group: Device Characteristics
 Best CV roc_auc: 0.4624 | Best params: {'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (64, 32)}

Running GridSearchCV for model: Logistic Regression on feature group: vision Characteristics
 Best CV roc_auc: 0.5402 | Best params: {'clf__C': 0.01}

Running GridSearch

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:37:42] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 Best CV roc_auc: 0.5569 | Best params: {'clf__learning_rate': 0.01, 'clf__max_depth': 6, 'clf__n_estimators': 100}

Running GridSearchCV for model: Gradient Boosting on feature group: vision Characteristics
 Best CV roc_auc: 0.5463 | Best params: {'clf__learning_rate': 0.01, 'clf__n_estimators': 200}

Running GridSearchCV for model: AdaBoost on feature group: vision Characteristics
 Best CV roc_auc: 0.5486 | Best params: {'clf__learning_rate': 1.0, 'clf__n_estimators': 100}

Running GridSearchCV for model: SVM (RBF) on feature group: vision Characteristics
 Best CV roc_auc: 0.5474 | Best params: {'clf__C': 0.1, 'clf__gamma': 'auto'}

Running GridSearchCV for model: MLP Neural Net on feature group: vision Characteristics
 Best CV roc_auc: 0.5374 | Best params: {'clf__alpha': 0.0001, 'clf__hidden_layer_sizes': (128, 64)}

=== GridSearchCV results (per feature group) ===
             Feature_Group                Model  Best_CV_score  \
18  Device Characteristics             AdaBoost     

# Ensembles: Stacking and Voting Classifiers: Stacking and Voting Classifiers

In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, StackingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception:
    xgb_available = False
try:
    from lightgbm import LGBMClassifier
    lgbm_available = True
except Exception:
    lgbm_available = False

# Optional sampling (SMOTE)
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE
    imblearn_available = True
except Exception:
    imblearn_available = False

from pandas.api.types import (
    is_integer_dtype, is_bool_dtype, is_object_dtype
)
from pandas import CategoricalDtype

RANDOM_STATE = 42

# Config
DATA_PATH = "6G_IoT_Beamforming_Dataset.csv"
TEST_SIZE = 0.2
N_SPLITS = 5  # for OOF threshold tuning

network_features = ['Frequency (GHz)', 'Transmit Power (dBm)', 'Bandwidth (MHz)',
                    'Codebook Size', 'Interference Level (dB)']

environment_features = ['Obstacle Density', 'Mobility (m/s)', 'Environment_Outdoor']

device_features = ['Number of Antennas', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

vision_features = ['SIFT Keypoints']

all_feature_groups = {
    "All Features": network_features + environment_features + device_features + vision_features,
    "Network Parameters": network_features,
    "Environmental Factors": environment_features,
    "Device Characteristics": device_features,
    "Vision Features": vision_features
}

target = 'Optimized'  # binary label 0/1

# Load and basic cleaning
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

missing_cols = [c for c in all_feature_groups["All Features"] + [target] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in CSV: {missing_cols}")

# Target to binary
y_raw = df[target]
y = pd.to_numeric(y_raw, errors='coerce')
if y.isna().any():
    y_map = (
        y_raw.astype(str).str.strip().str.lower()
        .map({'1': 1, '0': 0, 'yes': 1, 'no': 0, 'true': 1, 'false': 0,
              'optimized': 1, 'not optimized': 0, 'opt': 1, 'non-opt': 0})
    )
    if y_map.isna().any():
        raise ValueError("Target column cannot be coerced to binary 0/1. Please clean the target values.")
    y = y_map
y = y.astype(int)
if set(y.unique()) - {0, 1}:
    raise ValueError(f"Target must be binary 0/1. Found: {set(y.unique())}")

pos_rate = y.mean()
print(f"Class balance: positive rate = {pos_rate:.3f} ({y.sum()} positives out of {len(y)})")

# Column type detection
def detect_column_types(X_df, explicit_binary=None, explicit_categorical=None):
    explicit_binary = set(explicit_binary or [])
    explicit_categorical = set(explicit_categorical or [])

    binary_cols, categorical_cols, numeric_cols = [], [], []

    for col in X_df.columns:
        s = X_df[col]
        if col in explicit_binary:
            binary_cols.append(col); continue
        if col in explicit_categorical:
            categorical_cols.append(col); continue

        if is_bool_dtype(s):
            binary_cols.append(col)
        elif is_object_dtype(s) or isinstance(s.dtype, CategoricalDtype):
            categorical_cols.append(col)
        else:
            if is_integer_dtype(s):
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0, 1}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
            else:
                uniq = pd.unique(s.dropna())
                if set(uniq).issubset({0.0, 1.0}):
                    binary_cols.append(col)
                else:
                    numeric_cols.append(col)
    return numeric_cols, categorical_cols, binary_cols
explicit_binary = ['Environment_Outdoor', 'Device Type_IoT Sensor', 'Device Type_Smartphone']

# Preprocessing builders
# OneHotEncoder dense for compatibility with SMOTE
try:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('to_object', FunctionTransformer(lambda X: np.asarray(X).astype(object))),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', categorical_encoder)
])

binary_transformer = Pipeline(steps=[
    ('to_float', FunctionTransformer(lambda X: np.asarray(X).astype(float))),
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

def build_preprocessor(X_subset):
    num_cols, cat_cols, bin_cols = detect_column_types(X_subset, explicit_binary=explicit_binary)
    preproc = ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols),
        ('bin', binary_transformer, bin_cols)
    ], remainder='drop')
    return preproc

# Model zoo
def get_models(scale_pos_weight=1.0):
    # Define base estimators with imbalance handling
    lr_base = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
    rf_base = RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced_subsample', n_jobs=-1)
    gb_base = GradientBoostingClassifier(random_state=RANDOM_STATE) # Does not take class_weight
    ada_base = AdaBoostClassifier(n_estimators=400, learning_rate=0.05, random_state=RANDOM_STATE) # Does not take class_weight
    svc_base = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE)
    dt_base = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')
    knn_base = KNeighborsClassifier(n_neighbors=5) # Does not take class_weight
    extratrees_base = ExtraTreesClassifier(n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced_subsample', n_jobs=-1)
    histgb_base = HistGradientBoostingClassifier(random_state=RANDOM_STATE) # Does not take class_weight

    xgb_base = None
    if xgb_available:
        xgb_base = XGBClassifier(random_state=RANDOM_STATE, n_estimators=600, learning_rate=0.05,
                                max_depth=4, subsample=0.8, colsample_bytree=0.8,
                                reg_alpha=0.0, reg_lambda=1.0, n_jobs=-1,
                                eval_metric='logloss', scale_pos_weight=scale_pos_weight)

    lgbm_base = None
    if lgbm_available:
        lgbm_base = LGBMClassifier(random_state=RANDOM_STATE, n_estimators=600, learning_rate=0.05,
                                  num_leaves=31, max_depth=-1, colsample_bytree=0.8, subsample=0.8,
                                  reg_alpha=0.0, reg_lambda=1.0, n_jobs=-1,
                                  scale_pos_weight=scale_pos_weight)

    # Select a diverse set of base estimators for ensembles
    # These estimators still carry their own imbalance handling (class_weight/scale_pos_weight) or rely on SMOTE in the main pipeline.
    ensemble_base_estimators_for_stacking = [
        ('lr', lr_base),
        ('rf', rf_base),
        ('gb', gb_base),
    ]
    if xgb_base:
        ensemble_base_estimators_for_stacking.append(('xgb', xgb_base))
    if lgbm_base:
        ensemble_base_estimators_for_stacking.append(('lgbm', lgbm_base))
    ensemble_base_estimators_for_stacking = [e for e in ensemble_base_estimators_for_stacking if e[1] is not None]

    ensemble_base_estimators_for_voting = [
        ('lr', lr_base),
        ('rf', rf_base),
        ('svc', svc_base),
        ('histgb', histgb_base)
    ]
    if xgb_base:
        ensemble_base_estimators_for_voting.append(('xgb', xgb_base))
    if lgbm_base:
        ensemble_base_estimators_for_voting.append(('lgbm', lgbm_base))
    ensemble_base_estimators_for_voting = [e for e in ensemble_base_estimators_for_voting if e[1] is not None]

    models = {}

    if ensemble_base_estimators_for_stacking:
        stacking_clf = StackingClassifier(
            estimators=ensemble_base_estimators_for_stacking,
            final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), # Meta-learner
            cv=5,
            passthrough=True, # Pass original features AND base predictions to meta-learner
            n_jobs=-1
        )
        models["Stacking Ensemble"] = stacking_clf

    if ensemble_base_estimators_for_voting:
        voting_clf = VotingClassifier(
            estimators=ensemble_base_estimators_for_voting,
            voting='soft', # Use 'soft' for probability averaging
            n_jobs=-1
        )
        models["Voting Ensemble"] = voting_clf

    return models

# Threshold tuning via OOF predictions
def best_threshold_from_oof(pipeline, X_train, y_train, n_splits=5, metric="f1"):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros(len(y_train), dtype=float)

    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr = y_train.iloc[tr_idx]

        pipe = pipeline
        pipe.fit(X_tr, y_tr)

        if hasattr(pipe.named_steps['classifier'], "predict_proba"):
            proba = pipe.predict_proba(X_val)[:, 1]
        elif hasattr(pipe.named_steps['classifier'], "decision_function"):
            scores = pipe.decision_function(X_val)
            proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            proba = np.zeros(len(val_idx))
        oof_proba[val_idx] = proba

    y_true = y_train.values
    # Evaluate thresholds
    thresholds = np.unique(np.round(oof_proba, 4))
    if len(thresholds) < 50:
        thresholds = np.linspace(0.05, 0.95, 91)

    best_thr, best_val = 0.5, -1
    for thr in thresholds:
        y_hat = (oof_proba >= thr).astype(int)
        if metric == "f1":
            val = f1_score(y_true, y_hat, zero_division=0)
        elif metric == "recall":
            val = recall_score(y_true, y_hat, zero_division=0)
        else:
            val = f1_score(y_true, y_hat, zero_division=0)
        if val > best_val:
            best_val, best_thr = val, thr
    return float(best_thr), float(best_val)

# Evaluate function per group
def evaluate_group(df, features, y, group_name):
    X = df[features].copy()

    preproc = build_preprocessor(X)

    pos_rate = y.mean()
    neg_rate = 1 - pos_rate
    scale_pos_weight = (neg_rate / pos_rate) if 0 < pos_rate < 1 else 1.0

    models = get_models(scale_pos_weight=scale_pos_weight)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    use_smote = imblearn_available and pos_rate < 0.35  # enable if quite imbalanced
    results = []

    for name, clf in models.items():
        if use_smote:
            pipe_base = ImbPipeline(steps=[('preprocessor', preproc), ('smote', SMOTE(random_state=RANDOM_STATE)), ('classifier', clf)])
        else:
            pipe_base = Pipeline(steps=[('preprocessor', preproc), ('classifier', clf)])

        # Find threshold using OOF on training set (no test leakage)
        thr, oof_score = best_threshold_from_oof(pipe_base, X_train, y_train, n_splits=N_SPLITS, metric="f1")

        # Fit on full training and evaluate
        pipe_base.fit(X_train, y_train)

        # Probas for AUC metrics
        if hasattr(pipe_base.named_steps['classifier'], "predict_proba"):
            y_proba_test = pipe_base.predict_proba(X_test)[:, 1]
        elif hasattr(pipe_base.named_steps['classifier'], "decision_function"):
            scores = pipe_base.named_steps['classifier'].decision_function(X_test)
            y_proba_test = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
        else:
            y_proba_test = np.zeros_like(y_test, dtype=float)


        # Default 0.5
        y_pred_05 = (y_proba_test >= 0.5).astype(int)
        # Tuned threshold
        y_pred_thr = (y_proba_test >= thr).astype(int)


        # Metrics
        def safe_metrics(y_true, y_pred, y_proba):
            acc = accuracy_score(y_true, y_pred)
            prec = precision_score(y_true, y_pred, zero_division=0)
            rec = recall_score(y_true, y_pred, zero_division=0)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            roc = roc_auc_score(y_true, y_proba) if y_proba is not None and len(np.unique(y_true)) == 2 else np.nan
            pr  = average_precision_score(y_true, y_proba) if y_proba is not None and len(np.unique(y_true)) == 2 else np.nan
            return acc, prec, rec, f1, roc, pr

        acc05, prec05, rec05, f105, roc, prauc = safe_metrics(y_test, y_pred_05, y_proba_test)
        acct,   prect,   rect,   f1t,  _,   _  = safe_metrics(y_test, y_pred_thr, y_proba_test)


        results.append({
            "Feature_Group": group_name,
            "Model": name,
            "Use_SMOTE": use_smote,
            "Threshold_OOF_F1": round(thr, 3),
            "OOF_F1_at_Thr": round(oof_score, 3),
            "Accuracy@0.5": acc05,
            "Precision@0.5": prec05,
            "Recall@0.5": rec05,
            "F1@0.5": f105,
            "Accuracy@Tuned": acct,
            "Precision@Tuned": prect,
            "Recall@Tuned": rect,
            "F1@Tuned": f1t,
            "ROC-AUC": roc,
            "PR-AUC": prauc
        })


    return pd.DataFrame(results)

# Run evaluations
all_results = []
for group_name, features in all_feature_groups.items():
    res = evaluate_group(df, features, y, group_name)
    all_results.append(res)


final_results = pd.concat(all_results, ignore_index=True)

# Sort for readability: by Feature_Group then F1@Tuned desc
final_results = final_results.sort_values(by=["Feature_Group", "F1@Tuned"], ascending=[True, False])

print("\n=== Beamforming Optimization Classification Results (with preprocessing, imbalance handling, and threshold tuning) ===")
print(final_results)

final_results.to_csv("Beamforming_Classification_Results_Improved.csv", index=False)

Class balance: positive rate = 0.168 (168 positives out of 1000)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


=== Beamforming Optimization Classification Results (with preprocessing, imbalance handling, and threshold tuning) ===
            Feature_Group              Model  Use_SMOTE  Threshold_OOF_F1  \
1            All Features    Voting Ensemble       True             0.047   
0            All Features  Stacking Ensemble       True             0.036   
6  Device Characteristics  Stacking Ensemble       True             0.291   
7  Device Characteristics    Voting Ensemble       True             0.432   
4   Environmental Factors  Stacking Ensemble       True             0.060   
5   Environmental Factors    Voting Ensemble       True             0.141   
2      Network Parameters  Stacking Ensemble       True             0.104   
3      Network Parameters    Voting Ensemble       True             0.076   
9         Vision Features    Voting Ensemble       True             0.194   
8         Vision Features  Stacking Ensemble       True             0.165   

   OOF_F1_at_Thr  Accuracy@0.5  

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
